# Optimal sizing and location of a BESS for multiuse flexibility provision

In [126]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import datetime
import cvxpy as cp

In [127]:
df_network = pd.read_excel('Network_rural.xlsx')
df_community = pd.read_excel('community_rural.xlsx')

In [128]:
df_network

,fr,to,RL,XL,IMAX
0,b0,b1,0.005125,0.002553,4.571
1,b1,b2,0.005125,0.002553,4.571
2,b2,b3,0.005125,0.002553,4.571
3,b3,b4,0.005125,0.002553,4.571
4,b0,b5,0.005125,0.002553,4.571
5,b5,b6,0.005125,0.002553,4.571
6,b6,b7,0.005125,0.002553,4.571
7,b7,b8,0.005125,0.002553,4.571
8,b8,b9,0.005125,0.002553,4.571
9,b9,b10,0.005125,0.002553,4.571


In [129]:
price_da = pd.read_csv('day-ahead-FI-2021.csv')
price_up = pd.read_csv('up-prices.csv')
price_down = pd.read_csv('down-prices.csv')
freq1 = pd.read_csv('freq-jan-apr.csv')
freq2 = pd.read_csv('freq-may-aug.csv')
freq3 = pd.read_csv('freq-sep-dec.csv')

In [130]:
freq = [freq1, freq2, freq3]
df_freq = pd.concat(freq)

In [131]:
df_freq = df_freq.reset_index()

In [132]:
df_freq

,index,Alkuaika UTC,Lopetusaika UTC,Alkuaika UTC+03:00,Lopetusaika UTC+03:00,Taajuus - reaaliaikatieto
0,0,2020-12-31 21:01:00,2020-12-31 21:01:00,2021-01-01 00:01:00,2021-01-01 00:01:00,49.955
1,1,2020-12-31 21:04:00,2020-12-31 21:04:00,2021-01-01 00:04:00,2021-01-01 00:04:00,49.965
2,2,2020-12-31 21:07:00,2020-12-31 21:07:00,2021-01-01 00:07:00,2021-01-01 00:07:00,49.979
3,3,2020-12-31 21:10:00,2020-12-31 21:10:00,2021-01-01 00:10:00,2021-01-01 00:10:00,49.983
4,4,2020-12-31 21:13:00,2020-12-31 21:13:00,2021-01-01 00:13:00,2021-01-01 00:13:00,50.028
...,...,...,...,...,...,...
174982,58474,2021-12-31 20:46:00,2021-12-31 20:46:00,2021-12-31 23:46:00,2021-12-31 23:46:00,49.989
174983,58475,2021-12-31 20:49:00,2021-12-31 20:49:00,2021-12-31 23:49:00,2021-12-31 23:49:00,50.006
174984,58476,2021-12-31 20:52:00,2021-12-31 20:52:00,2021-12-31 23:52:00,2021-12-31 23:52:00,49.977
174985,58477,2021-12-31 20:55:00,2021-12-31 20:55:00,2021-12-31 23:55:00,2021-12-31 23:55:00,49.975


In [133]:
data = {'time': df_freq['Alkuaika UTC+03:00'],
        'frequency': df_freq['Taajuus - reaaliaikatieto']}

In [134]:
df = pd.DataFrame(data)

In [135]:
df['FC'] = 50 - df['frequency']

In [136]:
df.loc[df[df['FC']<-0.1].index,'FC'] = 0
df.loc[df[df['FC']>0.1].index,'FC'] = 0

In [137]:
df['time'] = pd.to_datetime(df['time'])

In [138]:
df['hour'] = df.time.dt.hour
df['daynu'] = df.time.dt.dayofyear - 1

In [139]:
df

,time,frequency,FC,hour,daynu
0,2021-01-01 00:01:00,49.955,0.045,0,0
1,2021-01-01 00:04:00,49.965,0.035,0,0
2,2021-01-01 00:07:00,49.979,0.021,0,0
3,2021-01-01 00:10:00,49.983,0.017,0,0
4,2021-01-01 00:13:00,50.028,-0.028,0,0
...,...,...,...,...,...
174982,2021-12-31 23:46:00,49.989,0.011,23,364
174983,2021-12-31 23:49:00,50.006,-0.006,23,364
174984,2021-12-31 23:52:00,49.977,0.023,23,364
174985,2021-12-31 23:55:00,49.975,0.025,23,364


In [140]:
data1 = {'time': price_up['Alkuaika UTC+03:00'],
        'price_up': price_up['Ylössäätöhinta säätösähkömarkkinoilla'],
        'price_down': price_down['Down-regulation price in the Balancing energy market'],
        'price_da': price_da['Day-ahead Price [EUR/MWh]'] }

In [141]:
price = pd.DataFrame(data1)

In [142]:
price['time'] = pd.to_datetime(price['time'])

In [143]:
price['hour'] = price.time.dt.hour
price['daynu'] = price.time.dt.dayofyear - 1

In [144]:
price

,time,price_up,price_down,price_da,hour,daynu
0,2021-01-01 00:00:00,26.55,18.50,26.55,0.0,0.0
1,2021-01-01 01:00:00,24.07,18.50,24.07,1.0,0.0
2,2021-01-01 02:00:00,24.95,18.50,24.95,2.0,0.0
3,2021-01-01 03:00:00,24.35,24.35,24.35,3.0,0.0
4,2021-01-01 04:00:00,23.98,23.98,23.98,4.0,0.0
...,...,...,...,...,...,...
8756,2021-12-31 20:00:00,98.21,98.21,105.07,20.0,364.0
8757,2021-12-31 21:00:00,73.98,73.98,98.21,21.0,364.0
8758,2021-12-31 22:00:00,44.95,33.96,73.98,22.0,364.0
8759,2021-12-31 23:00:00,51.03,51.03,33.96,23.0,364.0


## Modeling Distribution Network

In [145]:
df_network

,fr,to,RL,XL,IMAX
0,b0,b1,0.005125,0.002553,4.571
1,b1,b2,0.005125,0.002553,4.571
2,b2,b3,0.005125,0.002553,4.571
3,b3,b4,0.005125,0.002553,4.571
4,b0,b5,0.005125,0.002553,4.571
5,b5,b6,0.005125,0.002553,4.571
6,b6,b7,0.005125,0.002553,4.571
7,b7,b8,0.005125,0.002553,4.571
8,b8,b9,0.005125,0.002553,4.571
9,b9,b10,0.005125,0.002553,4.571


In [146]:
def line_parameters(n1,n2):
    i = (df_network.fr == str(n1)) & (df_network.to == str(n2))
    r = float(df_network.loc[df_network[i].index,'RL'])
    x = float(df_network.loc[df_network[i].index,'XL'])
    i_max = float(df_network.loc[df_network[i].index,'IMAX'])
    result = np.array([r,x,i_max])
    return(result)

In [147]:
def node_connector(n1):
    return(pd.concat([df_network[df_network.fr == str(n1)]['to'],df_network[df_network.to == str(n1)]['fr']]))

In [148]:
#Optimization problem
cons = []
obj = 0



In [149]:
df_community.columns.get_loc("b5")
df_community.columns.get_loc("b3")

3

In [150]:
df_community.iloc[:,range(1,11)] = df_community.iloc[:,range(1,11)] / 50

In [151]:
df_community

,Date_Hour_start,b1,b2,b3,b4,b5,b6,b7,b8,b9,b10
0,2019-01-01 00:00:00,0.19560,0.29396,0.14,0.27656,0.40554,0.14934,0.39176,0.16590,0.08638,0.12
1,2019-01-01 01:00:00,0.24816,0.21344,0.12,0.26752,0.34254,0.13152,0.33752,0.22776,0.12904,0.09
2,2019-01-01 02:00:00,0.16088,0.14852,0.14,0.21904,0.25438,0.13390,0.22896,0.21504,0.14042,0.12
3,2019-01-01 03:00:00,0.16724,0.13084,0.14,0.11044,0.16114,0.10008,0.21446,0.11952,0.10124,0.12
4,2019-01-01 04:00:00,0.19064,0.17432,0.14,0.14780,0.20386,0.07268,0.26964,0.16740,0.13296,0.12
...,...,...,...,...,...,...,...,...,...,...,...
8755,2019-12-31 19:00:00,0.19968,0.03472,0.26,0.07496,0.04424,0.26808,0.13456,0.43734,0.21582,1.17
8756,2019-12-31 20:00:00,0.19836,0.03172,0.22,0.04798,0.03182,0.25302,0.13090,0.23580,0.12594,1.11
8757,2019-12-31 21:00:00,0.19868,0.03152,0.22,0.04346,0.02114,0.17116,0.13086,0.25008,0.13486,0.84
8758,2019-12-31 22:00:00,0.23988,0.03732,0.22,0.15228,0.12054,0.14686,0.15726,0.33258,0.18536,0.39


In [152]:
df_community['Date_Hour_start'] = pd.to_datetime(df_community['Date_Hour_start'])

In [153]:
df_community['daynu'] = df_community.Date_Hour_start.dt.dayofyear - 1
df_community['hour'] = df_community.Date_Hour_start.dt.hour

In [154]:
node_list = pd.read_excel('Nodes.xlsx')


In [155]:
node_list

,node
0,b0
1,b1
2,b2
3,b3
4,b4
5,b5
6,b6
7,b7
8,b8
9,b9


In [156]:
def node_index(node):
    return(node_list[node_list['node']==node].index[0])

In [157]:
def node_name(n):
    return(node_list['node'][n])

In [158]:
node_index('b3')

3

In [159]:
node_name(4)

'b4'

In [160]:
node_number = len(node_list)
node_number

11

In [161]:
v_min = 0.95
v_max = 1.05
v_nom = 1

In [162]:
#f = np.ones(16)

In [163]:
#x = cp.Variable(shape = (ln,))
#np.asarray(df_network.fr == 'b1') @ x #matrix matrix multiplication

In [164]:
slack_bus = node_list.node[0] #determining the slack bus
slack_bus

'b0'

In [165]:
d = 1

In [166]:
T = len(df_community[df_community.daynu==d])
T

24

In [167]:
df_network

,fr,to,RL,XL,IMAX
0,b0,b1,0.005125,0.002553,4.571
1,b1,b2,0.005125,0.002553,4.571
2,b2,b3,0.005125,0.002553,4.571
3,b3,b4,0.005125,0.002553,4.571
4,b0,b5,0.005125,0.002553,4.571
5,b5,b6,0.005125,0.002553,4.571
6,b6,b7,0.005125,0.002553,4.571
7,b7,b8,0.005125,0.002553,4.571
8,b8,b9,0.005125,0.002553,4.571
9,b9,b10,0.005125,0.002553,4.571


In [168]:
df_scenario = pd.read_csv('scenario_s1.csv')

In [169]:
df_scenario

,Unnamed: 0,Date_Hour_start,b1,date,hour,ClusterNob1,ClusterNob2,b2,ClusterNob3,b3,...,ClusterNob7,b7,ClusterNob8,b8,ClusterNob9,b9,ClusterNob10,b10,Sum,daynu
0,0,2019-03-18 00:00:00,5.700,2019-03-18,0,3,1,8.692,0,11,...,0,11.542,2,13.116,2,7.319,3,6.0,94.360,76
1,1,2019-03-18 01:00:00,9.514,2019-03-18,1,3,1,7.626,0,10,...,0,12.383,2,5.043,2,3.426,3,7.5,79.803,76
2,2,2019-03-18 02:00:00,6.568,2019-03-18,2,3,1,6.152,0,10,...,0,9.436,2,6.492,2,4.647,3,6.0,67.170,76
3,3,2019-03-18 03:00:00,10.490,2019-03-18,3,3,1,5.316,0,11,...,0,10.561,2,8.175,2,5.635,3,6.0,68.934,76
4,4,2019-03-18 04:00:00,3.230,2019-03-18,4,3,1,4.050,0,10,...,0,5.665,2,7.191,2,5.100,3,6.0,58.359,76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,235,2019-08-07 19:00:00,0.526,2019-08-07,19,0,0,1.040,1,9,...,2,1.303,1,3.921,0,1.838,0,57.0,83.994,218
236,236,2019-08-07 20:00:00,1.712,2019-08-07,20,0,0,0.500,1,6,...,2,1.356,1,1.911,0,1.008,0,42.0,72.440,218
237,237,2019-08-07 21:00:00,1.902,2019-08-07,21,0,0,0.586,1,6,...,2,1.537,1,6.651,0,3.293,0,12.0,52.198,218
238,238,2019-08-07 22:00:00,4.624,2019-08-07,22,0,0,3.546,1,8,...,2,5.858,1,12.186,0,7.083,0,12.0,69.566,218


In [170]:
selected_day = df_scenario['daynu'].unique().tolist()

In [171]:
selected_day

[76, 114, 125, 141, 156, 172, 192, 199, 208, 218]

In [172]:
constr = []
objective = 0

In [173]:
node_index(slack_bus)

0

In [174]:
np.shape(np.array([df_network.fr == 'b2']).astype(int))

(1, 10)

In [175]:
#test section
#mmmm =  (np.array([df_network.fr == 'b1']).astype(int) @ flow_act_pos)[0,1]
#mmmm

In [176]:
np.shape(np.array([df_network.fr == 'b2']).astype(int) * np.array(df_network.RL))

(1, 10)

In [177]:
current_square = cp.Variable(shape = (len(df_network),T), nonneg = True)

In [178]:
#fff = ((np.array([df_network.fr == 'b2']).astype(int) * np.array(df_network.RL)) @ current_square)[0,23]
#fff

In [179]:
flow_act_pos = cp.Variable(shape = (len(df_network),T), nonneg = True)

In [180]:
df_scenario[df_scenario.daynu==selected_day[1]]

,Unnamed: 0,Date_Hour_start,b1,date,hour,ClusterNob1,ClusterNob2,b2,ClusterNob3,b3,...,ClusterNob7,b7,ClusterNob8,b8,ClusterNob9,b9,ClusterNob10,b10,Sum,daynu
24,24,2019-04-25 00:00:00,5.630,2019-04-25,0,0,0,9.462,0,10,...,2,12.277,1,7.218,0,4.007,3,7.5,73.151,114
25,25,2019-04-25 01:00:00,5.998,2019-04-25,1,0,0,3.178,0,10,...,2,6.177,1,4.518,0,2.147,3,6.0,46.201,114
26,26,2019-04-25 02:00:00,4.476,2019-04-25,2,0,0,2.276,0,10,...,2,4.514,1,4.512,0,2.570,3,6.0,39.506,114
27,27,2019-04-25 03:00:00,4.552,2019-04-25,3,0,0,2.148,0,11,...,2,4.424,1,5.631,0,2.924,3,7.5,44.478,114
28,28,2019-04-25 04:00:00,4.816,2019-04-25,4,0,0,4.994,0,10,...,2,7.402,1,5.298,0,2.459,3,6.0,47.418,114
29,29,2019-04-25 05:00:00,4.010,2019-04-25,5,0,0,5.438,0,12,...,2,7.443,1,5.367,0,2.916,3,6.0,49.609,114
30,30,2019-04-25 06:00:00,2.228,2019-04-25,6,0,0,2.206,0,13,...,2,3.320,1,9.678,0,4.802,3,10.5,55.266,114
31,31,2019-04-25 07:00:00,4.564,2019-04-25,7,0,0,1.414,0,33,...,2,3.696,1,8.883,0,4.937,3,12.0,75.239,114
32,32,2019-04-25 08:00:00,1.874,2019-04-25,8,0,0,1.212,0,58,...,2,2.149,1,7.398,0,5.526,3,10.5,90.994,114
33,33,2019-04-25 09:00:00,4.042,2019-04-25,9,0,0,1.000,0,71,...,2,3.021,1,1.098,0,0.993,3,24.0,107.298,114


In [181]:
v_min = 0.95
v_max = 1.05
v_nom = 1
q_factor = np.tan(np.arccos(0.9))
partitions = 10
max_flex = 100
delta_s = v_nom * df_network['IMAX'] / partitions
of_value = {}
opt_flex_up = {}
opt_flex_down = {}
opt_power_act_sb = {}
opt_power_react_sb = {}
opt_voltage_square = {}
opt_current_square = {}
opt_bess_place = {}
#for d in df_community.daynu.unique():      #Run the optimization for each day separately
j = -1
for d in selected_day:
    j = j + 1 
    const = []   
    t = 0
    objective = 0
    R = range(1,partitions+1)
    T = len(df_scenario[df_scenario.daynu==d])
    h = df_scenario[df_scenario.daynu==d]
    h = h.reset_index(drop=True)
    def load(n):
        return(np.array(h.get(node_list['node'][n], default=np.zeros(h.shape[0]))))

    #defining variables
    power_act_sb = cp.Variable(shape = (node_number,T)) #active power of slack bus
    power_react_sb = cp.Variable(shape = (node_number,T)) #reactive power of slack bus_
    flex_up = cp.Variable(shape = (node_number,T), nonneg = True) #upward flexibility of each node
    flex_down = cp.Variable(shape = (node_number,T), nonneg = True) #downward flexibility of each node
    flow_act_pos = cp.Variable(shape = (len(df_network),T), nonneg = True) #positive active power flow
    flow_act_neg = cp.Variable(shape = (len(df_network),T), nonneg = True) #negative active power flow
    flow_react_pos = cp.Variable(shape = (len(df_network),T), nonneg = True) #positive reactive power flow
    flow_react_neg = cp.Variable(shape = (len(df_network),T), nonneg = True) #negative reactive power flow
    voltage_square = cp.Variable(shape = (node_number,T), nonneg = True) #voltage square
    current_square = cp.Variable(shape = (len(df_network),T), nonneg = True) #current square
    bess_place = cp.Variable(shape = (node_number,), boolean= True)
    power_act_delta = np.zeros((partitions+1), dtype=type(cp.Variable()))
    for i in R:
        power_act_delta[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
    power_react_delta = np.zeros((partitions+1), dtype=type(cp.Variable()))
    for i in R:
        power_react_delta[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
    par1 = np.zeros((partitions+1), dtype=type(cp.Variable()))
    for i in R:
        par1[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
    par2 = np.zeros((partitions+1), dtype=type(cp.Variable()))
    for i in R:
        par2[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
    for t in range(T):
        for n in range(node_number):
            #Active power constraint

            const += [power_act_sb[n,t] - (load(n)[t] ) / 50 + flex_up[n,t]
            - flex_down[n,t] - (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_act_pos)[0,t]
            + (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_act_neg)[0,t]
            - ((np.array([df_network.fr == node_name(n)]).astype(int) 
            * np.array(df_network.RL)) @ current_square)[0,t]
            + (np.array([df_network.to == node_name(n)]).astype(int) @ flow_act_pos)[0,t]
            - (np.array([df_network.to == node_name(n)]).astype(int) @ flow_act_neg)[0,t] == 0]

            #Reactive power constraint
            const += [power_react_sb[n,t] - q_factor * (load(n)[t]) / 50
            - (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_react_pos)[0,t]
            + (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_react_neg)[0,t]
            - ((np.array([df_network.fr == node_name(n)]).astype(int) 
            * np.array(df_network.XL)) @ current_square)[0,t]
            + (np.array([df_network.to == node_name(n)]).astype(int) @ flow_react_pos)[0,t]
            - (np.array([df_network.to == node_name(n)]).astype(int) @ flow_react_neg)[0,t] == 0]

            #const += [power_act_sb[n,t] <= 16]

            if n != 0:
                const += [power_act_sb[n,t] == 0]
                const += [power_react_sb[n,t] == 0]
            
            #Voltage-related constraints
            const += [voltage_square[n,t] <= np.square(v_max)]
            const += [np.square(v_min) <= voltage_square[n,t]]

            #Flexible power-related constraints
            const += [flex_up[n,t] <= bess_place[n] * max_flex]
            const += [flex_down[n,t] <= bess_place[n] * max_flex]
            const += [voltage_square[1,t] == np.square(v_nom)]

        for net in range(len(df_network)):
         
            # voltage and power relationship:
            const += [voltage_square[node_index(df_network.loc[net,'fr']),t] -
            voltage_square[node_index(df_network.loc[net,'to']),t] -
            (np.square(df_network.RL[net]) + np.square(df_network.XL[net])) * current_square[net,t] -
            2 * df_network.RL[net] * (flow_act_pos[net,t] - flow_act_neg[net,t]) -
            2 * df_network.XL[net] * (flow_react_pos[net,t] - flow_react_neg[net,t]) == 0]

            #active and reactive power flow constraints
            const += [flow_act_pos[net,t] + flow_act_neg[net,t] <= v_nom * df_network.IMAX[net]]
            const += [flow_react_pos[net,t] + flow_react_neg[net,t] <= v_nom * df_network.IMAX[net]]

            #voltage constraint
            const += [current_square[net,t] <= np.square(df_network.IMAX[net])]

            #piece-wise linearization-related constraints
            
            const += [np.square(v_nom) * current_square[net,t] == 
            sum(par1[:])[net,t] + sum(par2[:])[net,t]]

            const += [flow_act_pos[net,t] + flow_act_neg[net,t] ==
            sum(power_act_delta[:])[net,t]]

            const += [flow_react_pos[net,t] + flow_react_neg[net,t] ==
            sum(power_react_delta[:])[net,t]]

            for r in R:
                const += [par1[r][net,t] == (2*r-1) * delta_s[net] * power_act_delta[r][net,t]]
                const += [par2[r][net,t] == (2*r-1) * delta_s[net] * power_react_delta[r][net,t]]
                const += [power_act_delta[r][net,t] <= delta_s[net]]
                const += [power_react_delta[r][net,t] <= delta_s[net]]

    #Binary variable-related constraint
    const += [cp.sum(bess_place[:]) <= 1]
#objective function
    objective = cp.sum(cp.sum(flex_up + flex_down))
    problem = cp.Problem(cp.Minimize(objective), const)

    #Saving variables
    of_value[j] = problem.solve(verbose=True)
    opt_flex_up[j] = flex_up.value
    opt_flex_down[j] = flex_down.value
    opt_power_act_sb[j] = power_act_sb.value
    opt_power_react_sb[j] = power_react_sb.value
    opt_voltage_square[j] = voltage_square.value
    opt_current_square[j] = current_square.value
    opt_bess_place[j] = bess_place.value

                                     CVXPY                                     
                                     v1.2.1                                    
(CVXPY) Apr 11 02:29:59 PM: Your problem has 12131 variables, 13609 constraints, and 0 parameters.
(CVXPY) Apr 11 02:30:00 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Apr 11 02:30:00 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Apr 11 02:30:00 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Apr 11 02:30:03 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Apr 11 02:30:03 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatri

In [182]:
opt_power_act_sb[8]

array([[ 0.79993583,  0.74512729,  0.85422455,  0.63737301,  0.62596919,
         0.5103283 ,  0.46705746,  0.40913419,  0.7753092 ,  0.5626983 ,
         0.6548155 ,  1.52198974,  1.63115034,  2.09784703,  1.98453938,
         1.98007267,  2.01230569,  2.14104823,  2.03295506,  2.03945731,
         1.24677298,  0.723472  ,  1.17783715,  1.12114539],
       [-0.        , -0.        , -0.        , -0.        , -0.        ,
        -0.        , -0.        , -0.        , -0.        , -0.        ,
        -0.        , -0.        , -0.        , -0.        , -0.        ,
        -0.        , -0.        , -0.        , -0.        , -0.        ,
        -0.        , -0.        , -0.        , -0.        ],
       [-0.        , -0.        , -0.        , -0.        , -0.        ,
        -0.        , -0.        , -0.        , -0.        , -0.        ,
        -0.        , -0.        , -0.        , -0.        , -0.        ,
        -0.        , -0.        , -0.        , -0.        , -0.        ,
  

In [183]:
of_value

{0: 0.6900781835533687,
 1: 0.0,
 2: 0.19740673808884823,
 3: 0.0,
 4: 0.23971468221721204,
 5: 0.12643825514797014,
 6: 0.7266116672496945,
 7: 0.028052383833310213,
 8: 0.7678208151228927,
 9: 0.12927281320329143}

In [199]:
opt_bess_place #No flexibility is needed for nodes 1 and 3

{0: array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 1: array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 2: array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 3: array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 4: array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 5: array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 6: array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 7: array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 8: array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 9: array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])}

In [193]:
#get the minimum capacity for the battery
max_up = []
max_down = []

for j in range(3,len(selected_day)):
        max_up.append(max(sum(opt_flex_up[j])))
        max_down.append(max(sum(opt_flex_down[j])))
m=max(max(max_up),max(max_down))
print(m)

0.47125480772458106


In [194]:
m*50

23.562740386229052